In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet(
    "../data/processed/unified_train_all_features.parquet"
)

val_df = pd.read_parquet(
    "../data/processed/unified_validation_all_features.parquet"
)

test_df = pd.read_parquet(
    "../data/processed/unified_test_all_features.parquet"
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (27326, 29)
Validation: (5839, 29)
Test: (5801, 29)


In [2]:
feature_columns = [
    "has_context",
    "question_length",
    "answer_length",
    "context_length",

    "entity_count",
    "person_count",
    "organization_count",
    "location_count",
    "date_count",

    "question_answer_similarity",
    "question_context_similarity",
    "answer_context_similarity",

    "retrieval_top1_similarity",
    "retrieval_top3_mean_similarity",
    "retrieval_coverage",

    "answer_length_normalized",
    "semantic_uncertainty",
    "retrieval_uncertainty",
    "feature_disagreement"
]

print("Number of features:", len(feature_columns))

Number of features: 19


In [3]:
for df in [train_df, val_df, test_df]:

    df["has_qa_similarity"] = (
        df["question_answer_similarity"]
        .notna()
        .astype(int)
    )

    df["has_context_similarity"] = (
        df["question_context_similarity"]
        .notna()
        .astype(int)
    )

In [4]:
feature_columns += [
    "has_qa_similarity",
    "has_context_similarity"
]

print("Total model features:", len(feature_columns))

Total model features: 21


In [22]:
X_train = train_df[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["label"].astype(int)
y_val = val_df["label"].astype(int)
y_test = test_df["label"].astype(int)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)

X_train: (27326, 21)
y_train: (27326,)
X_val: (5839, 21)
y_val: (5839,)
X_test: (5801, 21)


In [6]:
%pip install scikit-learn



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [8]:
baseline_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [9]:
baseline_model.fit(
    X_train,
    y_train
)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [10]:
train_prob = baseline_model.predict_proba(X_train)[:, 1]
val_prob = baseline_model.predict_proba(X_val)[:, 1]
test_prob = baseline_model.predict_proba(X_test)[:, 1]

In [11]:
test_pred = (
    test_prob >= 0.5
).astype(int)

In [12]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Test AUROC:",
      roc_auc_score(y_test, test_prob))

print("Test AUPRC:",
      average_precision_score(y_test, test_prob))

print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

print("Test Precision:",
      precision_score(y_test, test_pred))

print("Test Recall:",
      recall_score(y_test, test_pred))

print("Test F1:",
      f1_score(y_test, test_pred))

Test AUROC: 0.8101148236636594
Test AUPRC: 0.821665969838544
Test Accuracy: 0.7107395276676435
Test Precision: 0.6997713165632147
Test Recall: 0.7383660806618407
Test F1: 0.718550821871855


In [13]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    y_test,
    test_prob
)

print("Test Brier Score:", brier)

Test Brier Score: 0.17622882596634887


In [15]:
def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:
            mask = (
                (y_prob >= bins[i]) &
                (y_prob <= bins[i + 1])
            )
        else:
            mask = (
                (y_prob >= bins[i]) &
                (y_prob < bins[i + 1])
            )

        if not np.any(mask):
            continue

        bin_accuracy = np.mean(
            y_true[mask]
        )

        bin_confidence = np.mean(
            y_prob[mask]
        )

        bin_fraction = np.mean(mask)

        ece += (
            bin_fraction
            *
            abs(
                bin_accuracy -
                bin_confidence
            )
        )

    return ece

In [16]:
ece = expected_calibration_error(
    np.asarray(y_test),
    test_prob
)

print("Test ECE:", ece)

Test ECE: 0.03521211555844331


In [17]:
test_results = test_df[
    [
        "source_dataset",
        "label"
    ]
].copy()

test_results["probability"] = test_prob

In [18]:
for dataset_name in test_results["source_dataset"].unique():

    mask = (
        test_results["source_dataset"]
        == dataset_name
    )

    y_true_dataset = (
        test_results.loc[mask, "label"]
        .to_numpy()
    )

    y_prob_dataset = (
        test_results.loc[mask, "probability"]
        .to_numpy()
    )

    print("\nDataset:", dataset_name)

    print(
        "AUROC:",
        roc_auc_score(
            y_true_dataset,
            y_prob_dataset
        )
    )

    print(
        "AUPRC:",
        average_precision_score(
            y_true_dataset,
            y_prob_dataset
        )
    )


Dataset: truthfulqa
AUROC: 0.4761680957402908
AUPRC: 0.5190358380488196

Dataset: halueval
AUROC: 0.9469195555555556
AUPRC: 0.9541810634421584

Dataset: fever
AUROC: 0.5775382786406535
AUPRC: 0.5210847375030924


In [19]:
import joblib
import os

os.makedirs(
    "../models",
    exist_ok=True
)

joblib.dump(
    baseline_model,
    "../models/logistic_regression_baseline.pkl"
)

print("Baseline model saved.")

Baseline model saved.


In [23]:
feature_names = feature_columns

coefficients = (
    baseline_model
    .named_steps["model"]
    .coef_[0]
)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
)

print(
    feature_importance.to_string(index=False)
)

                       feature  coefficient  abs_coefficient
          semantic_uncertainty    -0.592872         0.592872
    question_answer_similarity     0.592872         0.592872
                 answer_length     0.585433         0.585433
          feature_disagreement    -0.457045         0.457045
                   has_context     0.280339         0.280339
        has_context_similarity     0.280339         0.280339
retrieval_top3_mean_similarity    -0.254861         0.254861
             has_qa_similarity    -0.210554         0.210554
               question_length     0.173284         0.173284
   question_context_similarity    -0.148608         0.148608
      answer_length_normalized     0.122598         0.122598
                context_length    -0.105077         0.105077
            organization_count    -0.089232         0.089232
                    date_count     0.086028         0.086028
                location_count    -0.063054         0.063054
     retrieval_top1_simi

In [24]:
dataset_sensitive_features = [
    "has_context",
    "question_context_similarity",
    "answer_context_similarity",
    "retrieval_top1_similarity",
    "retrieval_top3_mean_similarity",
    "retrieval_coverage",
    "retrieval_uncertainty"
]

In [25]:
robust_features = [
    f for f in feature_columns
    if f not in dataset_sensitive_features
]

print("Robust features:", len(robust_features))
print(robust_features)

Robust features: 14
['question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'question_answer_similarity', 'answer_length_normalized', 'semantic_uncertainty', 'feature_disagreement', 'has_qa_similarity', 'has_context_similarity']


In [26]:
baseline_results = pd.DataFrame({
    "model": ["logistic_regression"],
    "auroc": [roc_auc_score(y_test, test_prob)],
    "auprc": [average_precision_score(y_test, test_prob)],
    "accuracy": [accuracy_score(y_test, test_pred)],
    "precision": [precision_score(y_test, test_pred)],
    "recall": [recall_score(y_test, test_pred)],
    "f1": [f1_score(y_test, test_pred)],
    "brier": [brier_score_loss(y_test, test_prob)],
    "ece": [ece]
})

baseline_results

,model,auroc,auprc,accuracy,precision,recall,f1,brier,ece
0,logistic_regression,0.810115,0.821666,0.71074,0.699771,0.738366,0.718551,0.176229,0.035212


In [27]:
baseline_results.to_csv(
    "../experiments/baseline_results.csv",
    index=False
)

In [29]:
redundant_features = [
    "semantic_uncertainty",
    "retrieval_uncertainty",
    "has_context_similarity"
]

clean_feature_columns = [
    f for f in feature_columns
    if f not in redundant_features
]

print("Features before:", len(feature_columns))
print("Features after:", len(clean_feature_columns))
print(clean_feature_columns)

Features before: 21
Features after: 18
['has_context', 'question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'question_answer_similarity', 'question_context_similarity', 'answer_context_similarity', 'retrieval_top1_similarity', 'retrieval_top3_mean_similarity', 'retrieval_coverage', 'answer_length_normalized', 'feature_disagreement', 'has_qa_similarity']


In [30]:
model_a_features = clean_feature_columns

X_train_a = train_df[model_a_features].copy()
X_val_a = val_df[model_a_features].copy()
X_test_a = test_df[model_a_features].copy()

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model_a = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model_a.fit(X_train_a, y_train)

test_prob_a = model_a.predict_proba(X_test_a)[:, 1]

In [32]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    brier_score_loss
)

test_pred_a = (test_prob_a >= 0.5).astype(int)

print("AUROC:",
      roc_auc_score(y_test, test_prob_a))

print("AUPRC:",
      average_precision_score(y_test, test_prob_a))

print("Accuracy:",
      accuracy_score(y_test, test_pred_a))

print("F1:",
      f1_score(y_test, test_pred_a))

print("Brier:",
      brier_score_loss(y_test, test_prob_a))

AUROC: 0.8101105445209142
AUPRC: 0.8216517633259464
Accuracy: 0.7107395276676435
F1: 0.718550821871855
Brier: 0.1762361268384776


In [33]:
results_a = test_df[
    ["source_dataset", "label"]
].copy()

results_a["probability"] = test_prob_a

for dataset_name in results_a["source_dataset"].unique():

    mask = (
        results_a["source_dataset"]
        == dataset_name
    )

    y_true_dataset = (
        results_a.loc[mask, "label"]
        .to_numpy()
    )

    y_prob_dataset = (
        results_a.loc[mask, "probability"]
        .to_numpy()
    )

    print("\nDataset:", dataset_name)

    print(
        "AUROC:",
        roc_auc_score(
            y_true_dataset,
            y_prob_dataset
        )
    )

    print(
        "AUPRC:",
        average_precision_score(
            y_true_dataset,
            y_prob_dataset
        )
    )


Dataset: truthfulqa
AUROC: 0.4761680957402907
AUPRC: 0.5190747310170476

Dataset: halueval
AUROC: 0.9469337777777778
AUPRC: 0.9541975770166183

Dataset: fever
AUROC: 0.5773948701321644
AUPRC: 0.5209403512008455


In [34]:
robust_features = [
    "question_length",
    "answer_length",
    "context_length",
    "entity_count",
    "person_count",
    "organization_count",
    "location_count",
    "date_count",
    "question_answer_similarity",
    "answer_length_normalized",
    "semantic_uncertainty",
    "feature_disagreement",
    "has_qa_similarity",
    "has_context_similarity"
]

print("Robust feature count:", len(robust_features))

Robust feature count: 14


In [35]:
X_train_b = train_df[robust_features].copy()
X_val_b = val_df[robust_features].copy()
X_test_b = test_df[robust_features].copy()

In [36]:
model_b = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

model_b.fit(X_train_b, y_train)

test_prob_b = model_b.predict_proba(X_test_b)[:, 1]

In [37]:
test_pred_b = (
    test_prob_b >= 0.5
).astype(int)

print(
    "Overall AUROC:",
    roc_auc_score(y_test, test_prob_b)
)

print(
    "Overall AUPRC:",
    average_precision_score(y_test, test_prob_b)
)

print(
    "Accuracy:",
    accuracy_score(y_test, test_pred_b)
)

print(
    "F1:",
    f1_score(y_test, test_pred_b)
)

Overall AUROC: 0.8019971115786471
Overall AUPRC: 0.8130446648300016
Accuracy: 0.7002240992932253
F1: 0.7065969293065631


In [38]:
results_b = test_df[
    ["source_dataset", "label"]
].copy()

results_b["probability"] = test_prob_b

for dataset_name in results_b["source_dataset"].unique():

    mask = (
        results_b["source_dataset"]
        == dataset_name
    )

    y_true_dataset = (
        results_b.loc[mask, "label"].to_numpy()
    )

    y_prob_dataset = (
        results_b.loc[mask, "probability"].to_numpy()
    )

    print("\nDataset:", dataset_name)

    print(
        "AUROC:",
        roc_auc_score(
            y_true_dataset,
            y_prob_dataset
        )
    )

    print(
        "AUPRC:",
        average_precision_score(
            y_true_dataset,
            y_prob_dataset
        )
    )


Dataset: truthfulqa
AUROC: 0.4749300747397863
AUPRC: 0.5163447664949988

Dataset: halueval
AUROC: 0.9468440000000001
AUPRC: 0.9531842561275377

Dataset: fever
AUROC: 0.5294460274803956
AUPRC: 0.4899865060234165


In [39]:
for dataset_name in test_df["source_dataset"].unique():

    subset = test_df[
        test_df["source_dataset"] == dataset_name
    ].copy()

    print("\n" + "=" * 60)
    print("DATASET:", dataset_name)

    print("\nLabel distribution:")
    print(subset["label"].value_counts())

    # Match probabilities already generated by Model B
    mask = (
        test_df["source_dataset"]
        == dataset_name
    )

    probs = test_prob_b[mask.to_numpy()]

    print("\nMean predicted probability by label:")

    print(
        pd.DataFrame({
            "label": subset["label"].to_numpy(),
            "probability": probs
        })
        .groupby("label")["probability"]
        .mean()
    )


DATASET: truthfulqa

Label distribution:
label
1    452
0    386
Name: count, dtype: int64

Mean predicted probability by label:
label
0    0.558065
1    0.566197
Name: probability, dtype: float64

DATASET: halueval

Label distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Mean predicted probability by label:
label
0    0.231981
1    0.775076
Name: probability, dtype: float64

DATASET: fever

Label distribution:
label
0    1014
1     949
Name: count, dtype: int64

Mean predicted probability by label:
label
0    0.497260
1    0.503837
Name: probability, dtype: float64


In [40]:
for dataset_name in test_df["source_dataset"].unique():

    mask = (
        test_df["source_dataset"] == dataset_name
    ).to_numpy()

    y_true = y_test.to_numpy()[mask]
    y_prob = test_prob_b[mask]

    print("\n" + "=" * 60)
    print("DATASET:", dataset_name)
    print("Rows:", len(y_true))

    print(
        "AUROC:",
        roc_auc_score(y_true, y_prob)
    )

    print(
        "AUPRC:",
        average_precision_score(y_true, y_prob)
    )

    print("\nMean probability by label:")

    print(
        pd.DataFrame({
            "label": y_true,
            "probability": y_prob
        })
        .groupby("label")["probability"]
        .mean()
    )

    print("\nProbability range:")
    print(
        "min =", y_prob.min(),
        "max =", y_prob.max()
    )


DATASET: truthfulqa
Rows: 838
AUROC: 0.4749300747397863
AUPRC: 0.5163447664949988

Mean probability by label:
label
0    0.558065
1    0.566197
Name: probability, dtype: float64

Probability range:
min = 0.006626839930618529 max = 0.9578291369232195

DATASET: halueval
Rows: 3000
AUROC: 0.9468440000000001
AUPRC: 0.9531842561275377

Mean probability by label:
label
0    0.231981
1    0.775076
Name: probability, dtype: float64

Probability range:
min = 0.012423467600639376 max = 0.9999932206576526

DATASET: fever
Rows: 1963
AUROC: 0.5294460274803956
AUPRC: 0.4899865060234165

Mean probability by label:
label
0    0.497260
1    0.503837
Name: probability, dtype: float64

Probability range:
min = 0.2935511610971794 max = 0.9674914843755682


In [41]:
mask = (
    test_df["source_dataset"] == "halueval"
).to_numpy()

halueval_check = pd.DataFrame({
    "label": y_test.to_numpy()[mask],
    "probability": test_prob_b[mask]
})

print(
    halueval_check
    .sort_values("probability")
    .head(20)
)

print("\nHighest probabilities:")
print(
    halueval_check
    .sort_values("probability", ascending=False)
    .head(20)
)

      label  probability
1462      0     0.012423
554       0     0.012614
2652      0     0.015408
2088      0     0.015431
1060      0     0.015940
2796      0     0.016530
2310      0     0.016647
414       0     0.017271
300       0     0.017727
2032      0     0.018323
2188      0     0.018566
1976      0     0.018700
2528      0     0.019123
442       0     0.019132
278       0     0.019195
994       0     0.019350
1484      0     0.019410
1456      0     0.019445
406       0     0.020005
2370      0     0.020222

Highest probabilities:
      label  probability
920       0     0.999993
1007      1     0.999718
2343      1     0.997858
2785      1     0.996619
2743      1     0.996313
2021      1     0.995740
1303      1     0.995557
2233      1     0.995400
2849      1     0.995349
641       1     0.994846
663       1     0.993453
837       1     0.993242
583       1     0.993075
151       1     0.992999
2789      1     0.992917
2193      1     0.992804
1507      1     0.992767
1

In [42]:
h_train = train_df[
    train_df["source_dataset"] == "halueval"
].copy()

h_val = val_df[
    val_df["source_dataset"] == "halueval"
].copy()

h_test = test_df[
    test_df["source_dataset"] == "halueval"
].copy()

print("HaluEval train:", h_train.shape)
print("HaluEval val:", h_val.shape)
print("HaluEval test:", h_test.shape)

HaluEval train: (14000, 31)
HaluEval val: (3000, 31)
HaluEval test: (3000, 31)


In [43]:
X_h_train = h_train[clean_feature_columns]
y_h_train = h_train["label"].astype(int)

X_h_truthfulqa = test_df[
    test_df["source_dataset"] == "truthfulqa"
][clean_feature_columns]

y_truthfulqa = test_df[
    test_df["source_dataset"] == "truthfulqa"
]["label"].astype(int)

In [48]:
# HaluEval-only training data
h_train = train_df[
    train_df["source_dataset"] == "halueval"
].copy()

# TruthfulQA test data
truthfulqa_test = test_df[
    test_df["source_dataset"] == "truthfulqa"
].copy()

# FEVER test data
fever_test = test_df[
    test_df["source_dataset"] == "fever"
].copy()


# -----------------------------
# HaluEval training features
# -----------------------------
X_h_train = h_train[
    clean_feature_columns
].copy()

y_h_train = h_train["label"].astype(int)


# -----------------------------
# TruthfulQA test features
# -----------------------------
X_truthfulqa = truthfulqa_test[
    clean_feature_columns
].copy()

y_truthfulqa = truthfulqa_test[
    "label"
].astype(int)


# -----------------------------
# FEVER test features
# -----------------------------
X_fever = fever_test[
    clean_feature_columns
].copy()

y_fever = fever_test[
    "label"
].astype(int)


print("HaluEval train:", X_h_train.shape)
print("TruthfulQA test:", X_truthfulqa.shape)
print("FEVER test:", X_fever.shape)

HaluEval train: (14000, 18)
TruthfulQA test: (838, 18)
FEVER test: (1963, 18)


In [49]:
cross_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

cross_model.fit(
    X_h_train,
    y_h_train
)

print("HaluEval-only model trained.")

HaluEval-only model trained.


In [50]:
truthfulqa_prob = cross_model.predict_proba(
    X_truthfulqa
)[:, 1]

print(
    "HaluEval → TruthfulQA AUROC:",
    roc_auc_score(
        y_truthfulqa,
        truthfulqa_prob
    )
)

print(
    "HaluEval → TruthfulQA AUPRC:",
    average_precision_score(
        y_truthfulqa,
        truthfulqa_prob
    )
)

HaluEval → TruthfulQA AUROC: 0.46578820670365445
HaluEval → TruthfulQA AUPRC: 0.5156763127505511


In [51]:
fever_prob = cross_model.predict_proba(
    X_fever
)[:, 1]

print(
    "HaluEval → FEVER AUROC:",
    roc_auc_score(
        y_fever,
        fever_prob
    )
)

print(
    "HaluEval → FEVER AUPRC:",
    average_precision_score(
        y_fever,
        fever_prob
    )
)

HaluEval → FEVER AUROC: 0.5182014494651279
HaluEval → FEVER AUPRC: 0.48081014658364174


In [53]:
t_train = train_df[
    train_df["source_dataset"] == "truthfulqa"
].copy()

t_val = val_df[
    val_df["source_dataset"] == "truthfulqa"
].copy()

t_test = test_df[
    test_df["source_dataset"] == "truthfulqa"
].copy()

X_t_train = t_train[clean_feature_columns]
y_t_train = t_train["label"].astype(int)

X_t_test = t_test[clean_feature_columns]
y_t_test = t_test["label"].astype(int)

In [54]:
truthfulqa_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

truthfulqa_model.fit(
    X_t_train,
    y_t_train
)

/opt/homebrew/lib/python3.11/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['question_context_similarity' 'answer_context_similarity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['has_context','question_length','answer_length',..., 'answer_length_normalized','feature_disagreement','has_qa_similarity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace 

In [56]:
truthfulqa_prob = truthfulqa_model.predict_proba(
    X_t_test
)[:, 1]

print(
    "TruthfulQA-only AUROC:",
    roc_auc_score(y_t_test, truthfulqa_prob)
)

print(
    "TruthfulQA-only AUPRC:",
    average_precision_score(
        y_t_test,
        truthfulqa_prob
    )
)

TruthfulQA-only AUROC: 0.6283071209133844
TruthfulQA-only AUPRC: 0.6264086304024676


/opt/homebrew/lib/python3.11/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['question_context_similarity' 'answer_context_similarity']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [57]:
f_train = train_df[
    train_df["source_dataset"] == "fever"
].copy()

f_test = test_df[
    test_df["source_dataset"] == "fever"
].copy()

X_f_train = f_train[clean_feature_columns]
y_f_train = f_train["label"].astype(int)

X_f_test = f_test[clean_feature_columns]
y_f_test = f_test["label"].astype(int)

In [58]:
fever_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

fever_model.fit(
    X_f_train,
    y_f_train
)

/opt/homebrew/lib/python3.11/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['question_answer_similarity' 'question_context_similarity'
 'answer_context_similarity' 'feature_disagreement']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['has_context','question_length','answer_length',..., 'answer_length_normalized','feature_disagreement','has_qa_similarity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace 

In [59]:
fever_prob = fever_model.predict_proba(
    X_f_test
)[:, 1]

print(
    "FEVER-only AUROC:",
    roc_auc_score(y_f_test, fever_prob)
)

print(
    "FEVER-only AUPRC:",
    average_precision_score(
        y_f_test,
        fever_prob
    )
)

FEVER-only AUROC: 0.6649790187116928
FEVER-only AUPRC: 0.6238897963394069


/opt/homebrew/lib/python3.11/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['question_answer_similarity' 'question_context_similarity'
 'answer_context_similarity' 'feature_disagreement']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [60]:
def get_observed_features(df, feature_list):
    return [
        feature
        for feature in feature_list
        if df[feature].notna().any()
    ]

In [61]:
truthfulqa_features = get_observed_features(
    t_train,
    clean_feature_columns
)

print("TruthfulQA features:", len(truthfulqa_features))
print(truthfulqa_features)

TruthfulQA features: 16
['has_context', 'question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'question_answer_similarity', 'retrieval_top1_similarity', 'retrieval_top3_mean_similarity', 'retrieval_coverage', 'answer_length_normalized', 'feature_disagreement', 'has_qa_similarity']


In [62]:
fever_features = get_observed_features(
    f_train,
    clean_feature_columns
)

print("FEVER features:", len(fever_features))
print(fever_features)

FEVER features: 14
['has_context', 'question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'retrieval_top1_similarity', 'retrieval_top3_mean_similarity', 'retrieval_coverage', 'answer_length_normalized', 'has_qa_similarity']


In [63]:
def show_coefficients(model, features, title):
    coefficients = (
        model.named_steps["model"].coef_[0]
    )

    result = pd.DataFrame({
        "feature": features,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients)
    }).sort_values(
        "abs_coefficient",
        ascending=False
    )

    print("\n" + "=" * 60)
    print(title)
    print(result.to_string(index=False))

In [64]:
show_coefficients(
    truthfulqa_model,
    truthfulqa_features,
    "TruthfulQA Logistic Regression"
)


TruthfulQA Logistic Regression
                       feature  coefficient  abs_coefficient
    question_answer_similarity     0.669188         0.669188
          feature_disagreement    -0.387676         0.387676
      answer_length_normalized    -0.299801         0.299801
     retrieval_top1_similarity    -0.258932         0.258932
retrieval_top3_mean_similarity     0.224145         0.224145
               question_length     0.158004         0.158004
                  entity_count    -0.153166         0.153166
                 answer_length    -0.097516         0.097516
                    date_count     0.031965         0.031965
            retrieval_coverage    -0.030767         0.030767
                  person_count    -0.022562         0.022562
            organization_count     0.011471         0.011471
                location_count    -0.004283         0.004283
                   has_context     0.000000         0.000000
                context_length     0.000000         0

In [65]:
show_coefficients(
    fever_model,
    fever_features,
    "FEVER Logistic Regression"
)


FEVER Logistic Regression
                       feature  coefficient  abs_coefficient
retrieval_top3_mean_similarity    -0.537086         0.537086
      answer_length_normalized     0.454386         0.454386
               question_length    -0.244019         0.244019
                 answer_length    -0.244019         0.244019
                  entity_count     0.140769         0.140769
     retrieval_top1_similarity     0.129418         0.129418
                    date_count     0.057304         0.057304
            organization_count    -0.050568         0.050568
            retrieval_coverage     0.038298         0.038298
                  person_count    -0.023108         0.023108
                location_count     0.000741         0.000741
                   has_context     0.000000         0.000000
                context_length     0.000000         0.000000
             has_qa_similarity     0.000000         0.000000


In [ ]:
#TruthfulQA relies primarily on question–answer semantic alignment, while FEVER relies more heavily on retrieval/textual features,
# demonstrating substantial task-specific feature behavior.